In [1]:
import pandas as pd 
import numpy as np 

In [2]:
train_df = pd.read_csv('/Users/alextoy/Desktop/Kaggle Competitions /Housing Prices/house-prices-advanced-regression-techniques/train.csv')
test_df = pd.read_csv('/Users/alextoy/Desktop/Kaggle Competitions /Housing Prices/house-prices-advanced-regression-techniques/test.csv')

# Housing Price Machine Learning - Random Forest

In [8]:
#Import the requisite packages from sklearn 
#For splitting and cross validation
from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV, RandomizedSearchCV

#for the pipeling 
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

#evaluation metrics 
from sklearn.metrics import mean_squared_error, mean_absolute_error 

#to actually run the regressions 
#from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

In [4]:
#transform the dependent variable and get my train and test X 
y = np.log1p(train_df['SalePrice'])
X = train_df.drop(columns = ['SalePrice'])
X_test = test_df.copy()

In [5]:
#preprocess the data 
numeric_features = ['LotFrontage','LotArea','MasVnrArea','BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF','1stFlrSF',
                   '2ndFlrSF','LowQualFinSF','GrLivArea','BsmtFullBath','BsmtHalfBath','FullBath','HalfBath','BedroomAbvGr',
                   'KitchenAbvGr','TotRmsAbvGrd','Fireplaces','GarageCars','GarageArea','WoodDeckSF','OpenPorchSF','EnclosedPorch',
                   '3SsnPorch','ScreenPorch','PoolArea','MiscVal']

categorical_features = ['MSSubClass','MSZoning','Street','Alley','LotShape','LandContour','Utilities','LotConfig',
                       'LandSlope', 'Neighborhood', 'Condition1','Condition2','BldgType','HouseStyle','OverallQual',
                       'OverallCond','YearBuilt','YearRemodAdd','RoofStyle','RoofMatl','Exterior1st','Exterior2nd','MasVnrType',
                       'ExterQual','ExterCond','Foundation','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2',
                       'Heating','HeatingQC','CentralAir','Electrical','KitchenQual','Functional','FireplaceQu','GarageType','GarageYrBlt'
                       , 'GarageFinish', 'GarageQual','GarageCond','PavedDrive','PoolQC','MiscFeature','MoSold','YrSold','SaleType',
                       'SaleCondition']

numeric_transformer = Pipeline(steps = [("imputer",SimpleImputer(strategy = "median"))
                                      ])
categorical_transformer = Pipeline(steps = [("imputer", SimpleImputer(strategy = "constant", fill_value = "missing")),
                                            ("to_str", FunctionTransformer(lambda x: x.astype(str))),
                                           ("onehot", OneHotEncoder(handle_unknown = "ignore"))])

preprocess = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))
    

In [10]:
#define the base model
rf_model_tune = RandomForestRegressor(n_estimators = 800,
                                 random_state = 1234, 
                                 n_jobs = -1)

#define the pipeline
rf_model_tune_pipe = Pipeline(steps = [("preprocess", preprocess), 
                            ("model",rf_model_tune)])

# set the parameter space we want to search over
param_distributions = {
    "model__max_features":["sqrt", 0.5, 1.0],
    "model__min_samples_leaf":[1,2,5,10],
    "model__min_samples_split":[2,5,10,20],
    "model__max_depth":[None,10,20,40]
}

#set up a CV to search over for the grid search
cv = KFold(n_splits = 5, shuffle = True, random_state = 1234) 


#perform the grid search 
rf_search = RandomizedSearchCV(rf_model_tune_pipe,
                        param_distributions = param_distributions, 
                        scoring = "neg_root_mean_squared_error", 
                        cv = cv,
                        n_jobs = -1, 
                        verbose = 1)

#find the best random forest from the grid search
rf_search.fit(X,y)
best_rf = rf_search.best_estimator_

#define the scoring metric for the outer K-fold search
scoring = {
    "rmse":"neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error"
}

cv_results_rf_tuned = cross_validate(best_rf, X, y, cv=cv, scoring=scoring, return_train_score=False)


print("5-fold CV Random Forest RMSE:", (-cv_results_rf_tuned["test_rmse"]).mean(), "±", (-cv_results_rf_tuned["test_rmse"]).std())
print("5-fold CV Random Forest MAE: ", (-cv_results_rf_tuned["test_mae"]).mean(),  "±", (-cv_results_rf_tuned["test_mae"]).std())


Fitting 5 folds for each of 10 candidates, totalling 50 fits
5-fold CV Random Forest RMSE: 0.1501806476365914 ± 0.007328963699794291
5-fold CV Random Forest MAE:  0.10194268226930889 ± 0.0034723491338109055


In [11]:
# Now for submission to Kaggle 
best_rf.fit(X,y)
test_log_prediction = best_rf.predict(X_test)
test_prediction = np.expm1(test_log_prediction)

submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": test_prediction
})

submission.to_csv("/Users/alextoy/Desktop/Kaggle Competitions /Housing Prices/house-prices-advanced-regression-techniques/Submission_03022026_RandomForest.csv", index=False)
submission.head()

,Id,SalePrice
0,1461,130297.413952
1,1462,154759.338332
2,1463,181936.633655
3,1464,183124.423122
4,1465,195542.771434
